# K-nearest neighbours (KNN)
The breast cancer data, choosing k, decision surfaces and the failure cases.

In [1]:
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load the data: 569 tumours, 30 numeric measurements, target 0 = malignant, 1 = benign.
# (The original CSV also has an "id" column and an empty "Unnamed: 32" column; scikit-learn's copy has neither.)
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(X.shape)
print(y.value_counts())
X.head()

(569, 30)
target
1    357
0    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Split, scale, train, evaluate

In [2]:
# Hold back 20% of the rows as the test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
print(X_train.shape, X_test.shape)

# Learn the mean and standard deviation from the training set only, then apply them to both sets
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# k = 5 is the default number of neighbours
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)            # "training" only stores the rows
y_pred = knn.predict(X_test_s)
print("accuracy with scaling   ", accuracy_score(y_test, y_pred))

# The same model on unscaled data, for comparison
raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print("accuracy without scaling", accuracy_score(y_test, raw.predict(X_test)))

(455, 30) (114, 30)


accuracy with scaling    0.9736842105263158


accuracy without scaling 0.9122807017543859


## Choosing k
Heuristic: the square root of the number of training rows, made odd.

In [3]:
print("sqrt(n) =", np.sqrt(len(X_train)))

# Experiment: one model per k, scored on the test set (quick, but it lets the test set pick k)
ks = range(1, 16)
test_acc = [accuracy_score(y_test, KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train).predict(X_test_s)) for k in ks]
go.Figure(go.Scatter(x=list(ks), y=test_acc, mode="lines+markers")).update_layout(
    template="simple_white", xaxis_title="k", yaxis_title="test accuracy")

sqrt(n) = 21.330729007701542


In [4]:
# The correct way: choose k with cross-validation on the training set only, then test once.
# The pipeline re-fits the scaler inside every fold, so no fold sees the others' statistics.
model = make_pipeline(StandardScaler(), KNeighborsClassifier())
search = GridSearchCV(model, {"kneighborsclassifier__n_neighbors": list(ks)}, cv=5)
search.fit(X_train, y_train)
print("best k by CV:", search.best_params_, "CV accuracy", round(search.best_score_, 3))
print("final test accuracy:", search.score(X_test, y_test))

best k by CV: {'kneighborsclassifier__n_neighbors': 10} CV accuracy 0.971


final test accuracy: 0.9736842105263158


## Decision surface on two columns
The same code as `app.py` (run `python app.py` for the k slider).

In [5]:
from app import figure_for
figure_for(5)

## Failure case 1: prediction is slow on big data

In [6]:
rng = np.random.default_rng(0)
queries = rng.random((1000, 30))
for n in [10_000, 100_000, 500_000]:
    Xb, yb = rng.random((n, 30)), rng.integers(0, 2, n)
    m = KNeighborsClassifier(algorithm="brute")
    t0 = time.perf_counter(); m.fit(Xb, yb); t1 = time.perf_counter(); m.predict(queries); t2 = time.perf_counter()
    print(f"n = {n:>7,}: fit {1000 * (t1 - t0):6.1f} ms, predict 1000 rows {1000 * (t2 - t1):6.1f} ms")

n =  10,000: fit    1.6 ms, predict 1000 rows  688.4 ms


n = 100,000: fit   41.7 ms, predict 1000 rows 1422.0 ms


n = 500,000: fit   96.8 ms, predict 1000 rows 3075.8 ms


## Failure case 5: imbalanced classes

In [7]:
# 98% of rows in class 0, 2% in class 1
Xi, yi = make_classification(n_samples=5000, n_features=10, weights=[0.98], flip_y=0, class_sep=0.5, random_state=0)
a, b, c, d = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=0)
p = KNeighborsClassifier(n_neighbors=5).fit(a, c).predict(b)
print("rare-class rows in test:", d.sum())
print("accuracy", round(accuracy_score(d, p), 3), "| rare class found (recall)", round(recall_score(d, p), 3))

rare-class rows in test: 30
accuracy 0.98 | rare class found (recall) 0.033


## Tie-breaking in the vote

In [8]:
# A 2-against-2 tie with k = 4: scikit-learn returns the smaller class label, whichever class is "left" or "right"
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
Xt = np.array([[0.0], [1.0], [3.0], [4.0]])
for labels in ([0, 0, 1, 1], [1, 1, 0, 0]):
    print(labels, KNeighborsClassifier(n_neighbors=4).fit(Xt, labels).predict([[2.0]]))

[0, 0, 1, 1] [0]


[1, 1, 0, 0]

 [0]
